# Focused Direct and Total Site

**Learning outcome:** Apply focused direct and total site through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Core  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** `notebook`

## Study question and data

**Study question:** How does a threshold problem's equivalent global dt_min differ from ordinary direct, indirect, and Total Site targets?

## Step 1: Establish the site-wide reference

This cell loads `pulp_mill.json` and runs `problem.target.all_heat_integration()` across the whole site hierarchy. The result is stored as `complete`, and `summary` tabulates the targets. This site-wide solution is the reference for the focused scopes in the next step.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("pulp_mill.json", project_name="Site")
complete = problem.target.all_heat_integration()
summary = problem.summary_frame()
summary

## Step 2: Run explicit integration scopes and inspect the equivalent dt_min

`direct_heat_integration(zone="Bleaching", period_id="0")` targets one zone, and its recovery feeds `heat_recovery_dt_min()`. That inverse call returns the greatest global `dt_min` still achieving the requested recovery. `recovery_dt_min_summary` records whether cached results were preserved and whether the direct recovery equals the thermodynamic limit. The cell also runs indirect and Total Site targets and builds `total_site_profiles` and `site_utility_curve`.

In [ ]:
direct = problem.target.direct_heat_integration(zone="Bleaching", period_id="0")
direct_recovery_kw = float(direct.heat_recovery_target)
requested_recovery_kw = direct_recovery_kw
cached_results_before_inverse = problem.results
recovery_dt_min = problem.target.heat_recovery_dt_min(
    heat_recovery={"value": requested_recovery_kw, "unit": "kW"},
    zone="Bleaching",
    period_id="0",
)
ordinary_target_preserved = problem.results.model_dump(
    mode="json"
) == cached_results_before_inverse.model_dump(mode="json")
recovery_dt_min_summary = recovery_dt_min.model_dump(mode="json")
thermodynamic_limit_matches_direct = (
    abs(recovery_dt_min.thermodynamic_limit.value - direct_recovery_kw) <= 1e-6
)
recovery_dt_min_summary["ordinary_target_preserved"] = ordinary_target_preserved
recovery_dt_min_summary["direct_recovery_is_limit"] = thermodynamic_limit_matches_direct
indirect = problem.target.indirect_heat_integration()
total_site = problem.target.total_site_heat_integration()
total_site_profiles = problem.plot.total_site_profiles()
site_utility_curve = problem.plot.site_utility_grand_composite_curve()

## Review the result

Inspect the requested and achieved recovery, positive threshold approach, thermodynamic limit, residual, status, and iteration count together. The direct-recovery comparison confirms that the requested value is the maximum-recovery boundary, while the ordinary target identity demonstrates that the inverse call is non-mutating. Then compare the process summary with the Total Site profiles and utility grand composite curve; the site views reveal opportunities that are not visible within one process zone.

In [ ]:
from IPython.display import display

display(summary)
display(recovery_dt_min_summary)
display(total_site_profiles)
display(site_utility_curve)

## Interpret the result

Bleaching is a threshold problem: its ordinary direct recovery equals the thermodynamic limit over a positive global-approach plateau. The inverse therefore returns the greatest feasible approach on that plateau, about 58.34505 delta_degC, with `at_thermodynamic_limit`; it does not force the limit boundary to zero. This is process-level composite-curve spacing, not an exchanger EMAT. Total Site curves describe utility-system opportunities, not individual exchanger matches.

## Adapt this template

Change `zone="Bleaching"` to a path from your zone tree and compare the same scope across scenarios.